In [1]:
import seaborn as sns
import pandas as pd
sns.set(font_scale=1.5)
import matplotlib.pyplot as plt
import numpy as np
from sklearn import linear_model

import plotly.graph_objects as go
import plotly.express as px

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.utils import shuffle
from sklearn.linear_model import Ridge
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import GridSearchCV

In [2]:
from sklearn.datasets import load_iris

In [3]:
iris = load_iris()

In [4]:
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

In [30]:
len(df)

150

In [5]:
df['target'] = iris.target

In [6]:
df['species'] = df['target'].map({i: name for i, name in enumerate(iris.target_names)})

In [7]:
df.columns

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'target', 'species'],
      dtype='str')

In [8]:
numeric_features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)','petal width (cm)']

In [9]:
all_indices = range(0, len(df))
all_indices = shuffle(all_indices)
training_indices, dev_indices = np.split(all_indices, [320])
training_data = df.iloc[training_indices]
dev_data = df.iloc[dev_indices]

In [10]:
training_data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
109,7.2,3.6,6.1,2.5,2,virginica
18,5.7,3.8,1.7,0.3,0,setosa
141,6.9,3.1,5.1,2.3,2,virginica
95,5.7,3.0,4.2,1.2,1,versicolor
2,4.7,3.2,1.3,0.2,0,setosa


In [11]:
scaled_lasso_model = Pipeline([
    ('transform', PolynomialFeatures(degree = 3, include_bias = False)),
    ('scale', StandardScaler()),
    ('lasso', Lasso())
])

In [12]:
parameters_to_try = {'lasso__alpha': 10**np.linspace(-4, 4, 100)}

In [13]:
model_finder = GridSearchCV(estimator = scaled_lasso_model,
                               param_grid = parameters_to_try,
                               scoring = "neg_mean_squared_error",
                               cv=[[training_indices, dev_indices]])

In [14]:
model_finder.fit(df[numeric_features], df["target"])

/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.885e+00, tolerance: 1.000e-02
  model = cd_fast.enet_coordinate_descent(
/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_cal

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...o', Lasso())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.",{'lasso__alpha': array([1.0000...00000000e+04])}
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[[array([109, ... 9, 137, 114]), array([], dtype=int64)]]"
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and para

In [15]:
model_finder.best_estimator_.named_steps['lasso'].coef_

array([-0.48613228, -0.26227702,  0.18823296,  0.40527933,  0.06068697,
        0.34714414,  0.14840047,  0.24310125, -0.01129988, -0.38806514,
       -0.02042726,  0.50926467,  0.0874219 ,  0.54779835, -0.62693799,
        0.62858412, -0.12076736, -0.        ,  0.04101791,  0.1197778 ,
       -0.00443809, -0.        , -0.07947436,  0.14284748, -0.00674874,
       -0.56664954, -0.49965013,  1.08293603,  0.        ,  0.10782562,
       -0.11408976, -0.54382511, -0.20176373, -0.10231442])

In [16]:
best_model = model_finder.best_estimator_

In [17]:
#lasso_weights = pd.DataFrame([best_model.named_steps["lasso"].coef_],
#                             columns = best_model.named_steps["transform"].get_feature_names_out())
lasso_weights = pd.DataFrame([model_finder.best_estimator_.named_steps["lasso"].coef_],
                             columns=model_finder.best_estimator_.named_steps["transform"].get_feature_names_out())


In [18]:
lasso_weights

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),sepal length (cm)^2,sepal length (cm) sepal width (cm),sepal length (cm) petal length (cm),sepal length (cm) petal width (cm),sepal width (cm)^2,sepal width (cm) petal length (cm),...,sepal width (cm)^3,sepal width (cm)^2 petal length (cm),sepal width (cm)^2 petal width (cm),sepal width (cm) petal length (cm)^2,sepal width (cm) petal length (cm) petal width (cm),sepal width (cm) petal width (cm)^2,petal length (cm)^3,petal length (cm)^2 petal width (cm),petal length (cm) petal width (cm)^2,petal width (cm)^3
0,-0.486132,-0.262277,0.188233,0.405279,0.060687,0.347144,0.1484,0.243101,-0.0113,-0.388065,...,-0.006749,-0.56665,-0.49965,1.082936,0.0,0.107826,-0.11409,-0.543825,-0.201764,-0.102314


In [20]:
# Get all feature names and coefficients
feature_names = model_finder.best_estimator_ \
                            .named_steps['transform'] \
                            .get_feature_names_out()

coefficients = model_finder.best_estimator_ \
                           .named_steps['lasso'].coef_



In [21]:
coefficients

array([-0.48613228, -0.26227702,  0.18823296,  0.40527933,  0.06068697,
        0.34714414,  0.14840047,  0.24310125, -0.01129988, -0.38806514,
       -0.02042726,  0.50926467,  0.0874219 ,  0.54779835, -0.62693799,
        0.62858412, -0.12076736, -0.        ,  0.04101791,  0.1197778 ,
       -0.00443809, -0.        , -0.07947436,  0.14284748, -0.00674874,
       -0.56664954, -0.49965013,  1.08293603,  0.        ,  0.10782562,
       -0.11408976, -0.54382511, -0.20176373, -0.10231442])

In [22]:
pd.DataFrame({"features":feature_names, "coefficients":coefficients})

,features,coefficients
0,sepal length (cm),-0.486132
1,sepal width (cm),-0.262277
2,petal length (cm),0.188233
3,petal width (cm),0.405279
4,sepal length (cm)^2,0.060687
5,sepal length (cm) sepal width (cm),0.347144
6,sepal length (cm) petal length (cm),0.148400
7,sepal length (cm) petal width (cm),0.243101
8,sepal width (cm)^2,-0.011300
9,sepal width (cm) petal length (cm),-0.388065


In [25]:
coef_df = pd.DataFrame({"features": feature_names, "coefficients": coefficients})
sortdf = coef_df.sort_values(by="coefficients", key=abs, ascending=False)

In [26]:
sortdf

,features,coefficients
27,sepal width (cm) petal length (cm)^2,1.082936
15,sepal length (cm)^2 sepal width (cm),0.628584
14,sepal length (cm)^3,-0.626938
25,sepal width (cm)^2 petal length (cm),-0.566650
13,petal width (cm)^2,0.547798
31,petal length (cm)^2 petal width (cm),-0.543825
11,petal length (cm)^2,0.509265
26,sepal width (cm)^2 petal width (cm),-0.499650
0,sepal length (cm),-0.486132
3,petal width (cm),0.405279


In [28]:
feature_counts = pd.DataFrame({
    'feature': numeric_features,
    'non_zero_terms': [
        len(sortdf[(sortdf['coefficients'] != 0) & 
                   (sortdf['features'].str.contains(feature, regex=False))])
        for feature in numeric_features
    ]
}).sort_values('non_zero_terms', ascending=False)

In [29]:
feature_counts

,feature,non_zero_terms
1,sepal width (cm),14
0,sepal length (cm),13
2,petal length (cm),13
3,petal width (cm),13
